# Harmonic Features — jSymbolic-inspired Metrics
Métricas armónicas para el dataset `dataset.csv`.

In [2]:
import pandas as pd

df = pd.read_parquet("dataset.parquet", engine="fastparquet")

print(f"Total progresiones cargadas: {len(df)}")
print(f"Columnas: {list(df.columns)}")

print(f"\nEjemplo de chords: {df['chords'].iloc[0][:5]}")
print(f"Ejemplo de vector: {df['vectors'].iloc[0][0]}")

print(f"\nDistribución de longitudes:")
print(df['vectors'].map(len).describe())

Total progresiones cargadas: 1339089
Columnas: ['chords', 'vectors']

Ejemplo de chords: ['F', 'C', 'E7', 'Am', 'C']
Ejemplo de vector: [1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0]

Distribución de longitudes:
count    1.339089e+06
mean     1.907412e+01
std      2.357084e+01
min      1.000000e+00
25%      4.000000e+00
50%      1.000000e+01
75%      2.600000e+01
max      1.679000e+03
Name: vectors, dtype: float64


In [4]:
from itertools import combinations
import numpy as np

INTERVAL_NAMES = ['unísono', 'm2', 'M2', 'm3', 'M3', '4ta', 'tritono', '5ta', 'm6', 'M6', 'm7', 'M7']

# Intervalos disonantes por clase intervalar: m2(1), tritono(6), M7(11)
DISSONANT_INTERVALS = {1, 6, 11}

# MÉTRICA: Extraer intervalos de un vector de clase de pitch
# Definición: dado un vector binario de 12 posiciones (clases de pitch),
# se calculan todas las distancias mod-12 entre pares de notas activas.
# Cálculo: para cada par de PCs (p1, p2) activos, se calcula (p2-p1) % 12.
# Interpretación: revela la estructura intervalar vertical de cada acorde.
# Relación: inspirado en el "Vertical Interval Histogram" de jSymbolic.
def extract_intervals(vec):
    active = np.where(np.array(vec) > 0)[0]
    if len(active) < 2:
        return []
    return [int((p2 - p1) % 12) for p1, p2 in combinations(active, 2)]

# Perfiles de Krumhansl-Schmuckler para estimación de tonalidad
# (mismos coeficientes que usa music21 internamente)
MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
KEY_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

# Validación rápida con music21 para confirmar que los perfiles coinciden
from music21.analysis.floatingKey import KeyAnalyzer
from music21 import stream, note, key
print("\nPerfiles de Krumhansl-Schmuckler cargados correctamente.")
print("Validando con music21... (se usa un stream de prueba)")


Perfiles de Krumhansl-Schmuckler cargados correctamente.
Validando con music21... (se usa un stream de prueba)


In [5]:
# MÉTRICA: Disonancia vertical
# Definición: proporción de intervalos disonantes dentro de cada acorde,
# promediada a lo largo de toda la progresión.
# Cálculo: para cada acorde, se cuentan los intervalos presentes y se divide
# la cantidad de intervalos disonantes (m2=1, tritono=6, M7=11) por el total.
# Interpretación: valores altos indican acordes con mayor tensión sonora.
# Relación: inspirado en "Dissonant Interval Ratio" de jSymbolic.
def compute_dissonance(vectors):
    diss_ratios = []
    for vec in vectors:
        intervals = extract_intervals(vec)
        if not intervals:
            continue
        diss_count = sum(1 for i in intervals if i in DISSONANT_INTERVALS)
        diss_ratios.append(diss_count / len(intervals))
    return np.mean(diss_ratios) if diss_ratios else 0.0

print("=== DISONANCIA ===")
for i in range(3):
    vecs = df['vectors'].iloc[i]
    d = compute_dissonance(vecs)
    print(f"  Progresión {i}: disonancia = {d:.4f}")

=== DISONANCIA ===
  Progresión 0: disonancia = 0.0392
  Progresión 1: disonancia = 0.0269
  Progresión 2: disonancia = 0.0000


In [6]:
# MÉTRICA: Complejidad estructural de acordes
# Definición: mide la densidad armónica de los acordes de una progresión
# contando cuántas clases de pitch tiene cada acorde.
# Cálculo: se suman los valores del vector (0 o 1) para obtener la cardinalidad.
# Clasifica en: tríadas (3), séptimas (4), extendidos (>4).
# Interpretación: acordes más densos suelen percibirse como más complejos.
# Relación: inspirado en las features de estructura de acordes de jSymbolic.
def compute_chord_complexity(vectors):
    sizes = [sum(vec) for vec in vectors]
    if not sizes:
        return {'avg_size': 0, 'triad_ratio': 0, 'seventh_ratio': 0, 'extended_ratio': 0}
    n = len(sizes)
    return {
        'avg_size': np.mean(sizes),
        'triad_ratio': sum(1 for s in sizes if s == 3) / n,
        'seventh_ratio': sum(1 for s in sizes if s == 4) / n,
        'extended_ratio': sum(1 for s in sizes if s > 4) / n,
    }

print("=== COMPLEJIDAD ESTRUCTURAL ===")
for i in range(3):
    vecs = df['vectors'].iloc[i]
    cc = compute_chord_complexity(vecs)
    print(f"  Progresión {i}: avg_size={cc['avg_size']:.2f}, tríadas={cc['triad_ratio']:.2f}, séptimas={cc['seventh_ratio']:.2f}, extendidos={cc['extended_ratio']:.2f}")

=== COMPLEJIDAD ESTRUCTURAL ===
  Progresión 0: avg_size=3.24, tríadas=0.76, séptimas=0.24, extendidos=0.00
  Progresión 1: avg_size=3.16, tríadas=0.84, séptimas=0.16, extendidos=0.00
  Progresión 2: avg_size=3.00, tríadas=1.00, séptimas=0.00, extendidos=0.00


In [7]:
from scipy.stats import entropy as shannon_entropy

# MÉTRICA: Distribución de intervalos y entropía
# Definición: histograma de clases intervalares (0-11) presentes en toda
# la progresión, y entropía de Shannon de esa distribución.
# Cálculo: se extraen todos los intervalos de todos los acordes, se construye
# un histograma de 12 bins, y se aplica H = -Σ p(x)·log2(p(x)).
# Interpretación: mayor entropía = mayor variedad intervalar = mayor complejidad.
# Relación: inspirado en las features de distribución intervalar de jSymbolic.
def compute_interval_distribution(vectors):
    all_intervals = []
    for vec in vectors:
        all_intervals.extend(extract_intervals(vec))
    hist = np.zeros(12, dtype=int)
    for i in all_intervals:
        hist[i] += 1
    total = hist.sum()
    if total == 0:
        return hist, 0.0
    probs = hist[hist > 0] / total
    ent = shannon_entropy(probs, base=2)
    return hist, ent

print("=== DISTRIBUCIÓN DE INTERVALOS ===")
for i in range(3):
    vecs = df['vectors'].iloc[i]
    hist, ent = compute_interval_distribution(vecs)
    print(f"  Progresión {i}: entropía = {ent:.4f}")
    print(f"    Histograma: {hist}")

=== DISTRIBUCIÓN DE INTERVALOS ===
  Progresión 0: entropía = 2.6552
    Histograma: [ 0  0  4 11 17  8  4  9  0 10  0  0]
  Progresión 1: entropía = 2.7159
    Histograma: [ 0  0  5 21 28 17  5 14  3 15  0  0]
  Progresión 2: entropía = 2.1972
    Histograma: [0 0 0 3 2 1 0 2 1 0 0 0]


In [8]:
# MÉTRICA: Variabilidad armónica
# Definición: mide cuánta diversidad de acordes y conjuntos de pitch-class
# hay en una progresión.
# Cálculo: unique_chords / total_chords, unique_pc_sets / total_chords,
# y desviación estándar del tamaño de los acordes.
# Interpretación: progresiones con alta variedad son armónicamente más ricas.
# Relación: inspirado en métricas de variedad melódica y armónica de jSymbolic.
def compute_variability(chords, vectors):
    total = len(chords)
    if total == 0:
        return {'unique_chord_ratio': 0, 'unique_pc_ratio': 0, 'size_std': 0}
    unique_chords = len(set(chords))
    unique_pc = len(set(tuple(v) for v in vectors))
    sizes = [sum(v) for v in vectors]
    return {
        'unique_chord_ratio': unique_chords / total,
        'unique_pc_ratio': unique_pc / total,
        'size_std': np.std(sizes),
    }

print("=== VARIABILIDAD ARMÓNICA ===")
for i in range(3):
    ch = df['chords'].iloc[i]
    vecs = df['vectors'].iloc[i]
    v = compute_variability(ch, vecs)
    print(f"  Progresión {i}: unique_chord={v['unique_chord_ratio']:.3f}, unique_pc={v['unique_pc_ratio']:.3f}, size_std={v['size_std']:.3f}")

=== VARIABILIDAD ARMÓNICA ===
  Progresión 0: unique_chord=0.294, unique_pc=0.294, size_std=0.424
  Progresión 1: unique_chord=0.355, unique_pc=0.355, size_std=0.368
  Progresión 2: unique_chord=1.000, unique_pc=1.000, size_std=0.000


In [9]:
# MÉTRICA: Features de transición
# Definición: analiza las transiciones entre acordes consecutivos de la progresión.
# Cálculo:
#   - Diversidad: número de transiciones únicas / total de transiciones.
#   - Entropía: entropía de Shannon sobre la distribución de transiciones.
#   - Distancia PC promedio: distancia Hamming entre vectores consecutivos.
# Interpretación: transiciones variadas e impredecibles indican mayor complejidad.
# Relación: inspirado en modelos secuenciales MIR y cadenas de Markov armónicas.
def compute_transitions(chords, vectors):
    n = len(vectors)
    if n < 2:
        return {'transition_diversity': 0, 'transition_entropy': 0, 'avg_pc_distance': 0}
    transitions = [(chords[i], chords[i + 1]) for i in range(n - 1)]
    distances = [np.sum(np.abs(np.array(vectors[i]) - np.array(vectors[i + 1]))) for i in range(n - 1)]
    unique = len(set(transitions))
    total = len(transitions)
    diversity = unique / total
    counts = {}
    for t in transitions:
        counts[t] = counts.get(t, 0) + 1
    probs = np.array(list(counts.values())) / total
    trans_entropy = shannon_entropy(probs, base=2)
    return {
        'transition_diversity': diversity,
        'transition_entropy': trans_entropy,
        'avg_pc_distance': np.mean(distances),
    }

print("=== TRANSICIONES ===")
for i in range(3):
    ch = df['chords'].iloc[i]
    vecs = df['vectors'].iloc[i]
    t = compute_transitions(ch, vecs)
    print(f"  Progresión {i}: diversity={t['transition_diversity']:.3f}, entropy={t['transition_entropy']:.4f}, avg_dist={t['avg_pc_distance']:.2f}")

=== TRANSICIONES ===
  Progresión 0: diversity=0.500, entropy=2.9056, avg_dist=4.25
  Progresión 1: diversity=0.600, entropy=3.9647, avg_dist=4.20
  Progresión 2: diversity=1.000, entropy=1.0000, avg_dist=5.00


In [10]:
# MÉTRICA: Features tonales (Krumhansl-Schmuckler)
# Definición: estima la tonalidad de la progresión y mide la estabilidad tonal.
# Cálculo:
#   - Se acumula un histograma global de pitch classes de toda la progresión.
#   - Se correlaciona con los perfiles de Krumhansl-Schmuckler (major/minor)
#     desplazados cíclicamente por las 12 tonalidades.
#   - Se selecciona la tonalidad con mayor correlación.
#   - La varianza tonal se calcula como la varianza de las correlaciones
#     entre cada acorde individual y el histograma global de PCs.
# Interpretación: baja varianza = acordes más estables dentro de la tonalidad;
#   alta varianza = progresiones que se alejan del centro tonal.
# Relación: basado en el algoritmo de Krumhansl-Schmuckler (1982),
#   implementado en music21 y usado en jSymbolic para análisis tonal.
#   Se usa la implementación directa (numpy) por rendimiento:
#   evitar crear 1.3M streams de music21.
def compute_tonal_features(vectors):
    n = len(vectors)
    if n == 0:
        return {'estimated_key': '', 'tonal_variance': 0.0}
    global_pc = np.zeros(12)
    for vec in vectors:
        global_pc += np.array(vec)
    scores = {}
    for i in range(12):
        major_corr = np.corrcoef(global_pc, np.roll(MAJOR_PROFILE, i))[0, 1]
        minor_corr = np.corrcoef(global_pc, np.roll(MINOR_PROFILE, i))[0, 1]
        scores[f'{KEY_NAMES[i]} major'] = major_corr
        scores[f'{KEY_NAMES[i]} minor'] = minor_corr
    best_key = max(scores, key=scores.get)
    correlations = []
    for vec in vectors:
        arr = np.array(vec)
        if np.std(arr) > 0 and np.std(global_pc) > 0:
            corr = np.corrcoef(arr, global_pc)[0, 1]
            correlations.append(corr)
        else:
            correlations.append(0.0)
    return {
        'estimated_key': best_key,
        'tonal_variance': np.var(correlations) if correlations else 0.0,
    }

print("=== FEATURES TONALES ===")
for i in range(3):
    vecs = df['vectors'].iloc[i]
    t = compute_tonal_features(vecs)
    print(f"  Progresión {i}: key={t['estimated_key']}, tonal_var={t['tonal_variance']:.4f}")

=== FEATURES TONALES ===
  Progresión 0: key=C major, tonal_var=0.0783
  Progresión 1: key=G major, tonal_var=0.0272
  Progresión 2: key=A major, tonal_var=0.0158


In [11]:
# MÉTRICA: Repetición / Periodicidad
# Definición: detecta patrones repetitivos en la secuencia de acordes.
# Cálculo: se busca el menor divisor k de N tal que la secuencia de N acordes
# se construye repitiendo el prefijo de longitud k. Si no hay, k = N.
# Repetición = k / N. Si el período es corto, hay mucha repetición.
# Interpretación: progresiones cíclicas (típicas en pop) tienen baja periodicidad
#   (ratio cercano a 0), mientras que progresiones no repetitivas tienen ratio=1.
# Relación: análisis de patrones usado en MIR para detección de estructura formal.
def compute_periodicity(chords):
    n = len(chords)
    if n == 0:
        return {'period_length': 0, 'repetition_ratio': 0}
    for k in range(1, n + 1):
        if n % k != 0:
            continue
        pattern = chords[:k]
        if pattern * (n // k) == chords:
            return {
                'period_length': k,
                'repetition_ratio': k / n,
            }
    return {'period_length': n, 'repetition_ratio': 1.0}

print("=== REPETICIÓN / PERIODICIDAD ===")
for i in range(3):
    ch = df['chords'].iloc[i]
    p = compute_periodicity(ch)
    print(f"  Progresión {i}: period={p['period_length']}, repetition_ratio={p['repetition_ratio']:.4f}")

=== REPETICIÓN / PERIODICIDAD ===
  Progresión 0: period=17, repetition_ratio=1.0000
  Progresión 1: period=31, repetition_ratio=1.0000
  Progresión 2: period=3, repetition_ratio=1.0000


In [12]:
# Función principal: calcula todas las métricas para una progresión
def compute_all_metrics(row):
    chords = row['chords']
    vectors = row['vectors']
    metrics = {}
    metrics['dissonance'] = compute_dissonance(vectors)
    cc = compute_chord_complexity(vectors)
    metrics['avg_chord_size'] = cc['avg_size']
    metrics['triad_ratio'] = cc['triad_ratio']
    metrics['seventh_ratio'] = cc['seventh_ratio']
    metrics['extended_ratio'] = cc['extended_ratio']
    _, interval_ent = compute_interval_distribution(vectors)
    metrics['interval_entropy'] = interval_ent
    var = compute_variability(chords, vectors)
    metrics['unique_chord_ratio'] = var['unique_chord_ratio']
    metrics['unique_pc_ratio'] = var['unique_pc_ratio']
    metrics['chord_size_std'] = var['size_std']
    trans = compute_transitions(chords, vectors)
    metrics['transition_diversity'] = trans['transition_diversity']
    metrics['transition_entropy'] = trans['transition_entropy']
    metrics['avg_pc_distance'] = trans['avg_pc_distance']
    tonal = compute_tonal_features(vectors)
    metrics['estimated_key'] = tonal['estimated_key']
    metrics['tonal_variance'] = tonal['tonal_variance']
    per = compute_periodicity(chords)
    metrics['period_length'] = per['period_length']
    metrics['repetition_ratio'] = per['repetition_ratio']
    return metrics

print("Calculando métricas para primeras 5 progresiones...")
for i in range(5):
    m = compute_all_metrics(df.iloc[i])
    print(f"\nProgresión {i}:")
    for k, v in m.items():
        print(f"  {k}: {v}")

Calculando métricas para primeras 5 progresiones...

Progresión 0:
  dissonance: 0.0392156862745098
  avg_chord_size: 3.235294117647059
  triad_ratio: 0.7647058823529411
  seventh_ratio: 0.23529411764705882
  extended_ratio: 0.0
  interval_entropy: 2.6552287795614964
  unique_chord_ratio: 0.29411764705882354
  unique_pc_ratio: 0.29411764705882354
  chord_size_std: 0.42418250299576343
  transition_diversity: 0.5
  transition_entropy: 2.9056390622295662
  avg_pc_distance: 4.25
  estimated_key: C major
  tonal_variance: 0.07834193881273312
  period_length: 17
  repetition_ratio: 1.0

Progresión 1:
  dissonance: 0.026881720430107524
  avg_chord_size: 3.161290322580645
  triad_ratio: 0.8387096774193549
  seventh_ratio: 0.16129032258064516
  extended_ratio: 0.0
  interval_entropy: 2.7158877957635243
  unique_chord_ratio: 0.3548387096774194
  unique_pc_ratio: 0.3548387096774194
  chord_size_std: 0.3677985242255284
  transition_diversity: 0.6
  transition_entropy: 3.9647351787255047
  avg_pc_d

In [13]:
# MÉTRICA: Composite Harmonic Complexity Score (CHCS)
# Definición: puntuación compuesta que combina disonancia, diversidad
# e imprevisibilidad de las transiciones en un único score.
# Cálculo:
#   1. Normalización min-max de todas las métricas numéricas.
#   2. Agrupación en 3 ejes:
#      - Dissonance: dissonance, interval_entropy
#      - Diversity: unique_chord_ratio, unique_pc_ratio, transition_diversity
#      - Variability: chord_size_std, transition_entropy, avg_pc_distance
#   3. Promedio ponderado: CHCS = 0.35*dissonance + 0.35*diversity + 0.30*variability
# Interpretación: mayor CHCS = progresión armónicamente más compleja.
#   El score oscila entre 0 (máxima simplicidad) y 1 (máxima complejidad).
def compute_chcs(metrics_df):
    df_norm = metrics_df.copy()
    def minmax_normalize(series):
        min_v, max_v = series.min(), series.max()
        if max_v == min_v:
            return series * 0 + 0.5
        return (series - min_v) / (max_v - min_v)
    df_norm['dissonance_norm'] = minmax_normalize(df_norm['dissonance'])
    df_norm['interval_entropy_norm'] = minmax_normalize(df_norm['interval_entropy'])
    df_norm['unique_chord_ratio_norm'] = minmax_normalize(df_norm['unique_chord_ratio'])
    df_norm['unique_pc_ratio_norm'] = minmax_normalize(df_norm['unique_pc_ratio'])
    df_norm['transition_diversity_norm'] = minmax_normalize(df_norm['transition_diversity'])
    df_norm['chord_size_std_norm'] = minmax_normalize(df_norm['chord_size_std'])
    df_norm['transition_entropy_norm'] = minmax_normalize(df_norm['transition_entropy'])
    df_norm['avg_pc_distance_norm'] = minmax_normalize(df_norm['avg_pc_distance'])
    df_norm['score_dissonance'] = (df_norm['dissonance_norm'] + df_norm['interval_entropy_norm']) / 2
    df_norm['score_diversity'] = (df_norm['unique_chord_ratio_norm'] + df_norm['unique_pc_ratio_norm'] + df_norm['transition_diversity_norm']) / 3
    df_norm['score_variability'] = (df_norm['chord_size_std_norm'] + df_norm['transition_entropy_norm'] + df_norm['avg_pc_distance_norm']) / 3
    df_norm['CHCS'] = 0.35 * df_norm['score_dissonance'] + 0.35 * df_norm['score_diversity'] + 0.30 * df_norm['score_variability']
    return df_norm

print("=== CHCS (Composite Harmonic Complexity Score) ===")
print("Definido como: 0.35*dissonance + 0.35*diversity + 0.30*variability")

=== CHCS (Composite Harmonic Complexity Score) ===
Definido como: 0.35*dissonance + 0.35*diversity + 0.30*variability


In [14]:
from tqdm import tqdm

tqdm.pandas()
print("Procesando todas las progresiones... esto puede tardar varios minutos.")

metrics_df = df.copy()
metrics = metrics_df.progress_apply(compute_all_metrics, axis=1, result_type='expand')
metrics_df = pd.concat([metrics_df, metrics], axis=1)

# Calcular CHCS
metrics_df = compute_chcs(metrics_df)

print(f"\nFilas procesadas: {len(metrics_df)}")
print(f"\nFila de ejemplo:")
sample_cols = ['chords', 'dissonance', 'interval_entropy', 'unique_chord_ratio',
               'transition_entropy', 'estimated_key', 'tonal_variance', 'CHCS']
print(metrics_df.iloc[0][sample_cols])
print(f"\nEstadísticas resumen:")
summary_cols = ['dissonance', 'interval_entropy', 'unique_chord_ratio',
                'transition_entropy', 'avg_pc_distance', 'tonal_variance', 'CHCS']
print(metrics_df[summary_cols].describe().round(4))

Procesando todas las progresiones... esto puede tardar varios minutos.


  3%|▎         | 46025/1339089 [03:55<1:54:21, 188.45it/s]/home/pepebeats/SCL_2.0/.venv/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/pepebeats/SCL_2.0/.venv/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
  8%|▊         | 103966/1339089 [08:51<1:45:19, 195.44it/s]


KeyboardInterrupt: 

In [ ]:
output_cols = ['chords', 'dissonance', 'avg_chord_size', 'triad_ratio',
               'seventh_ratio', 'extended_ratio', 'interval_entropy',
               'unique_chord_ratio', 'unique_pc_ratio', 'chord_size_std',
               'transition_diversity', 'transition_entropy', 'avg_pc_distance',
               'estimated_key', 'tonal_variance', 'period_length',
               'repetition_ratio', 'CHCS']

metrics_df[output_cols].to_parquet("metrics.parquet", index=False)
print(f"Guardado metrics.parquet con {len(metrics_df)} filas y {len(output_cols)} columnas.")